# GrokForge — SFT Baseline (Colab)

Fine-tunes `Qwen2.5-Coder-1.5B-Instruct` on the `skbose/grokforge-v1` dataset to generate Logstash Grok patterns from log lines.

**Runtime**: set to **T4 GPU** (Runtime → Change runtime type → T4)  
**Expected duration**: ~1–2 hours for 500 steps

## 1. GPU check

In [ ]:
!nvidia-smi

## 2. Install dependencies

In [ ]:
!pip install -q \
    transformers>=4.40.0 \
    datasets>=2.19.0 \
    trl>=0.8.6 \
    peft>=0.10.0 \
    bitsandbytes>=0.43.0 \
    accelerate>=0.29.0 \
    pygrok>=1.0.0 \
    huggingface_hub \
    wandb

## 3. Authenticate

Both services need credentials before training starts.

In [ ]:
# HuggingFace — write token required to push the trained model
# Get one at https://huggingface.co/settings/tokens
from huggingface_hub import login
login()

In [ ]:
# Weights & Biases — API key required to log metrics
# Get one at https://wandb.ai/settings
import wandb
wandb.login()

## 4. Configuration

All hyperparameters in one place — edit here before running training.  
Change `wandb_run_name` to distinguish experiments (e.g. `exp001-lr1e4`, `exp002-r32`).

In [ ]:
CFG = {
    # model & data
    "model_id":        "Qwen/Qwen2.5-Coder-1.5B-Instruct",
    "dataset_id":      "skbose/grokforge-v1",
    "output_model_id": "skbose/grokforge-sft-baseline",

    # QLoRA
    "lora_r":       16,
    "lora_alpha":   32,
    "lora_dropout": 0.05,

    # training
    "max_seq_length": 512,
    "batch_size":     8,
    "grad_accum":     2,    # effective batch = 16
    "lr":             1e-4,
    "max_steps":      500,
    "warmup_steps":   50,
    "save_steps":     100,
    "eval_steps":     100,
    "output_dir":     "./outputs",

    # wandb
    "wandb_project":  "grokforge",
    "wandb_run_name": "exp001-sft-baseline",
}

## 5. Load dataset

In [ ]:
from datasets import load_dataset

ds = load_dataset(CFG["dataset_id"])
print(ds)
print("\nSample:", ds["train"][0])

## 6. Load tokenizer & format dataset

Each `{log, pattern}` record is converted to Qwen's ChatML format using `apply_chat_template`.  
Sequences are short (p95 ≈ 120 tokens), well under the 512-token limit.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(CFG["model_id"])
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

SYSTEM = (
    "You are an expert at writing Logstash Grok patterns. "
    "Given a log line, output only the Grok pattern that matches it."
)

def format_example(example):
    messages = [
        {"role": "system",    "content": SYSTEM},
        {"role": "user",      "content": f"Log: {example['log']}"},
        {"role": "assistant", "content": example["pattern"]},
    ]
    return {
        "text": tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
    }

train_ds = ds["train"].map(format_example, remove_columns=ds["train"].column_names)
val_ds   = ds["val"].map(format_example,   remove_columns=ds["val"].column_names)

print(train_ds)
print("\nFormatted example:\n", train_ds[0]["text"])

## 7. Load model with QLoRA (4-bit)

- 4-bit NF4 quantization keeps the base model at ~1 GB on GPU
- LoRA adapters are trained in bfloat16 on top of the frozen base

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    CFG["model_id"],
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model.config.use_cache = False

lora_config = LoraConfig(
    r=CFG["lora_r"],
    lora_alpha=CFG["lora_alpha"],
    lora_dropout=CFG["lora_dropout"],
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

print(f"Model loaded. GPU memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 8. Train

`train_on_responses_only` masks the system + user tokens (labels → -100) so cross-entropy loss is computed only on the grok pattern tokens. This replaces the older `DataCollatorForCompletionOnlyLM` API removed in TRL 0.12+.

Checkpoints are pushed to the Hub every `save_steps` — if the session disconnects, progress is not lost.  
Training loss, eval loss, learning rate, and gradient norm are logged to W&B automatically.

In [ ]:
import os
from trl import SFTTrainer, SFTConfig, train_on_responses_only

os.environ["WANDB_PROJECT"] = CFG["wandb_project"]

sft_config = SFTConfig(
    output_dir=CFG["output_dir"],
    per_device_train_batch_size=CFG["batch_size"],
    gradient_accumulation_steps=CFG["grad_accum"],
    learning_rate=CFG["lr"],
    lr_scheduler_type="cosine",
    warmup_steps=CFG["warmup_steps"],
    max_steps=CFG["max_steps"],
    max_seq_length=CFG["max_seq_length"],
    dataset_text_field="text",
    eval_strategy="steps",
    eval_steps=CFG["eval_steps"],
    save_strategy="steps",
    save_steps=CFG["save_steps"],
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    push_to_hub=True,
    hub_model_id=CFG["output_model_id"],
    hub_strategy="checkpoint",
    logging_steps=10,
    bf16=True,
    report_to="wandb",
    run_name=CFG["wandb_run_name"],
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    peft_config=lora_config,
)

# Mask loss on system + user tokens; only train on the assistant's grok pattern.
# instruction_template marks where the prompt starts, response_template where the answer starts.
trainer = train_on_responses_only(
    trainer,
    instruction_template="<|im_start|>user\n",
    response_template="<|im_start|>assistant\n",
)

trainer.train()

## 9. Evaluation — exact match & functional match

Two metrics on the val split:
- **Exact match**: predicted pattern == ground truth pattern (string equality)
- **Functional match**: predicted pattern compiles and matches the original log line via `pygrok`

Both are logged to the active W&B run for comparison across experiments.

In [ ]:
from pygrok import Grok

model.eval()

def generate_pattern(log_line: str, max_new_tokens: int = 128) -> str:
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user",   "content": f"Log: {log_line}"},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output_ids[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def functional_match(log_line: str, pattern: str) -> bool:
    try:
        g = Grok(pattern)
        return g.match(log_line) is not None
    except Exception:
        return False


val_raw = ds["val"]
exact_hits = 0
func_hits  = 0
n = len(val_raw)

for i, example in enumerate(val_raw):
    pred = generate_pattern(example["log"])
    gt   = example["pattern"]

    if pred == gt:
        exact_hits += 1
    if functional_match(example["log"], pred):
        func_hits += 1

    if (i + 1) % 20 == 0:
        print(f"[{i+1}/{n}] exact={exact_hits/(i+1):.1%}  functional={func_hits/(i+1):.1%}")

exact_match_pct    = exact_hits / n
functional_match_pct = func_hits / n

print(f"\n=== Final ({n} examples) ===")
print(f"Exact match:      {exact_match_pct:.1%}  ({exact_hits}/{n})")
print(f"Functional match: {functional_match_pct:.1%}  ({func_hits}/{n})")

# Log to the active W&B run so these appear alongside training curves
wandb.log({
    "eval/exact_match":      exact_match_pct,
    "eval/functional_match": functional_match_pct,
})

## 10. Push final model to Hub & finish W&B run

In [ ]:
# Push best checkpoint (selected by load_best_model_at_end=True)
trainer.push_to_hub()
tokenizer.push_to_hub(CFG["output_model_id"])

print(f"Model available at: https://huggingface.co/{CFG['output_model_id']}")

wandb.finish()